# Lesson 29 · Common distributions — three faces of randomness: uniform, Gaussian, Bernoulli, and the PDF / CDF

← **Previous lesson** · [L28 · Mean, variance, standardization](L28_descriptive_stats.ipynb)

> Last lesson covered **mean, variance, standardization**: descriptive statistics, z-scores and comparing distributions.
> This lesson explores **common probability distributions**.

## This lesson's story: why are neural-network weights initialized from a normal distribution?

> **Main path (no overload)**: get to know the **normal** distribution well first (bell-shaped, centered on the mean, σ sets the width); for uniform and Bernoulli, just remember one "face" each.
> Success means: you can read a PDF/CDF plot and draw samples — you are **not** expected to prove everything from the axioms.

Ever wondered where a neural network's initial weights come from when training starts?

The answer is the **normal (Gaussian) distribution**: `He initialization` draws the weights from N(0, √(2/n)), so that the signal neither explodes nor vanishes as it passes from layer to layer. The maths behind that choice is the heart of this lesson — the shapes and properties of probability distributions.

The three most common distributions:

| Distribution | Parameters | Typical use |
|---|---|---|
| **Uniform distribution Uniform(a,b)** | lower bound a, upper bound b | random seeds, random offsets in data augmentation |
| **Normal distribution Normal(μ,σ)** | mean μ, standard deviation σ | weight initialization, noise models, error distributions |
| **Bernoulli distribution Bernoulli(p)** | success probability p | Dropout mask (each neuron is kept with probability p) |

In this lesson you implement `gaussian_pdf(x, mu, sigma)` — the probability density function (PDF) of the normal distribution — and check by numerical integration that the PDF integrates to 1 over ±∞.

## 1. Into the lab: how sampling converges to the shape of a distribution

Draw 5,000 samples from a uniform and from a normal distribution and look at the outline of each histogram: with few samples the shape is rough; with many it closes in on the theoretical PDF curve.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(0)
u = rng.uniform(-3, 3, 5000)    # uniform: every value in the interval equally likely
g = rng.normal(0, 1, 5000)      # normal: many in the middle, few at the edges
fig, ax = plt.subplots(1, 2, figsize=(9,3))
ax[0].hist(u, bins=40); ax[0].set_title('uniform')
ax[1].hist(g, bins=40); ax[1].set_title('normal'); plt.show()

## Hands-on: run random code more than once

Run the cell below several times and watch how much the mean of a small sample (N=10) jumps around each time, and how steady the mean of a large sample (N=10000) is.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    u = rng.uniform(0, 1, size=n)
    print(f'n={n:6d}  uniform sample mean={np.mean(u):.3f}  '  # theory: 0.5
          f'std={np.std(u):.3f}')           # theory: 1/sqrt(12)≈0.289


## Code experiment: repeat the same random experiment many times

This experiment shows that the bigger the sample, the less the mean varies from one repetition to the next — a direct demonstration of the law of large numbers (LLN).

#### The law of large numbers (LLN): why does the sample mean converge?

What the code below shows — the larger the sample size n, the closer together the means from different seeds (different random initializations) — is the **law of large numbers**.

**Intuition:** flip a coin 10 times and you might get 6 heads (sample mean = 0.6); flip it 1000 times and you get about 500 heads (sample mean ≈ 0.5); flip it a million times and you get closer still to 0.5.

**Why does this happen?**

Every draw carries some "random wobble", but when the sample is large those wobbles cancel each other out. In maths:

$$\text{Var}(\bar{X}_n) = \frac{\sigma^2}{n}$$

where $\bar{X}_n$ is the average of n samples and $\sigma$ is the standard deviation of a single sample.

**Key observation:** as n grows, $\frac{\sigma^2}{n}$ goes to 0, which means $\bar{X}_n$ varies less and less. In other words, the sample means from different seeds all cluster around the same point (the true μ).

**Numbers:**
- n=20: variance ≈ σ²/20, standard deviation ≈ σ/4.47 (about 4.47× tighter than a single sample)
- n=200: variance ≈ σ²/200, standard deviation ≈ σ/14.14 (smaller wobble)
- n=2000: variance ≈ σ²/2000, standard deviation ≈ σ/44.7 (hardly any wobble)

That is why, in machine learning, the more data you have, the more stable the model's parameter estimates, and the less any single sample's noise matters.

**Law of large numbers vs the central limit theorem (CLT):**
- **Law of large numbers**: the sample mean tends to the true mean (it's about the "center")
- **Central limit theorem**: the distribution of the sample mean tends to a normal distribution (it's about the "shape")

Both look like "more data, more regularity", but they look at it from different angles.

---

#### Going further: it's not just the mean that wobbles — counts wobble too

Above we saw that the wobble in the **mean** shrinks as the sample grows. But there is another common confusion: if "the 68-95-99.7 rule says only 0.27% of a normal distribution lies outside ±3σ", then out of 1000 samples, will **exactly** 1000×0.27% ≈ 2.7 fall outside?

**Answer: not "exactly" — "on average".**

The 68-95-99.7 percentages are **theoretical proportions** over **long, many-times-repeated** runs — just like "in the long run a coin lands heads 50% of the time". But for any **specific, finite** batch of samples (say, exactly 1000), the actual number that falls outside is a **random variable**, and it wobbles around its expected value of 2.7 — 0 this time, 5 next time, both completely normal. This "a specific batch deviates from the long-run theoretical value" effect is what statistics calls **sampling variation**.

It's two sides of the same coin as the law of large numbers:
- The **law of large numbers** says the "average" or "proportion" converges more and more steadily to the theoretical value as n grows (the wobble shrinks ∝ 1/√n).
- But for any **fixed, finite** n (say n=1000), the observed count still wobbles at random — it's just that the larger n is, the smaller that wobble is **as a share** of the total.

The code below checks this directly: how does the count "outside ±3σ" in 1000 samples actually wobble? This is also the real answer behind whiteboard challenge Q5 (MFCC features).

In [ ]:
import numpy as np

for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        samples = rng.normal(0, 1, size=n)  # standard normal
        estimates.append(np.mean(samples))
    print(f'n={n:4d} -> sample mean, 5 runs:', np.round(estimates, 3),
          'average=', round(float(np.mean(estimates)), 3))


#### Checking in code: how does the count "outside ±3σ" wobble?

In theory, a standard normal has only a 0.27% chance of landing outside ±3σ. If you draw 1000 samples each time, you "expect" about 1000×0.0027 ≈ 2.7 to fall outside. But **every individual draw comes out differently** — below we use 10 different random seeds, draw n=1000 samples with each, and count how many fall outside each time:

In [ ]:
import numpy as np

print("Draw n=1000 standard-normal samples each time and count |x|>3 (theoretical expectation ≈2.7)\n")
counts = []
for seed in range(10):
    rng = np.random.default_rng(seed)
    samples = rng.normal(0, 1, size=1000)
    count_outside = int(np.sum(np.abs(samples) > 3))
    counts.append(count_outside)
    print(f"seed={seed}: count outside = {count_outside}")

print(f"\ncounts over the 10 runs: {counts}")
print(f"mean count = {np.mean(counts):.2f} (should be close to the theoretical 2.7)")
print(f"std of the count = {np.std(counts):.2f} (this is the size of the wobble)")
print("\n💡 Conclusion: the count from a single draw wobbles around 2.7 (anything from 0 to 5 is normal);")
print("   this is 'sampling variation' — not a calculation error, but a built-in property of finite samples.")


## 2. The normal (Gaussian) distribution: the most important continuous distribution

**Why is the normal distribution everywhere?**

The central limit theorem (CLT): the average of enough independent random variables tends to a normal distribution — whatever the shape of the original distribution. Audio noise, measurement error, neural-network gradients… all are roughly normal.

**Probability density function (PDF)**:

$$f(x;\mu,\sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

**Key properties**:
- **Peak**: at x=μ, peak = 1/(σ√2π) ≈ 0.3989 (when σ=1)
- **Width**: the larger σ, the wider and lower the curve; the area is always 1
- **68-95-99.7 rule**: μ±1σ holds 68.27%, μ±2σ holds 95.45%, μ±3σ holds 99.73%

**Worked example** (standard normal μ=0, σ=1, x=0):
```
f(0) = 1/(1·√(2π)) = 1/√(6.283...) ≈ 0.3989
```

Double σ (σ=2) and the peak halves: f(0) = 1/(2√2π) ≈ 0.1995 — but the curve is wider, and the area is still 1.

> **Implementation hint**: `np.exp(-((x-mu)/sigma)**2 / 2) / (sigma * np.sqrt(2*np.pi))`

### The three ingredients of the normal formula, taken apart

$$f(x;\mu,\sigma) = \underbrace{\frac{1}{\sigma\sqrt{2\pi}}}_{\text{normalizing constant}} \quad \underbrace{\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)}_{\text{bell curve}}$$

**Part 1: the exponential $\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$**

This sets the **shape** of the curve. Key observations:
- $(x-\mu)^2$: the farther from the center, the larger it gets. The minus sign in the exponent makes the density die away quickly far from the center
- $2\sigma^2$ in the denominator: the larger σ, the slower the decay (the wider the curve)

**Why a square and not an absolute value?** The square $(x-\mu)^2$ is "smoother" than $|x-\mu|$ and much easier to differentiate and manipulate mathematically (which matters a lot for ML optimization).

**Why $2\sigma^2$ and not $\sigma^2$ or $3\sigma^2$?** The 2 is what makes σ² come out as exactly the variance of the distribution (with $\sigma^2$ alone, the variance would be $\sigma^2/2$). It isn't an arbitrary choice: keeping the parameter equal to the true variance is what lets the variance rule for sums of independent variables, $\text{Var}(X_1+X_2) = \text{Var}(X_1) + \text{Var}(X_2)$, apply to it directly.

**Part 2: the normalizing constant $\frac{1}{\sigma\sqrt{2\pi}}$**

This sets the **height** of the curve, and it exists for one reason only: **to make the area integrate to 1**.

With just the exponential and no constant, the area under the curve would be some arbitrary number (not 1). With the constant, the integral is always 1, however large σ is.

**Why $\sqrt{2\pi}$ and not something else?** It's a "gift" from mathematics: for the Gaussian function, $\int_{-\infty}^{\infty} e^{-x^2} dx = \sqrt{\pi}$ is a classic result. Integrate $e^{-x^2/(2\sigma^2)}$ with a change of variables and $\sqrt{2\pi}\sigma$ falls out naturally. So the normalizing constant is $\frac{1}{\sqrt{2\pi}\sigma}$.

**Checking the numbers: doubling σ really does halve the peak**

Peak of the standard normal (σ=1):
$$f(0; \mu=0, \sigma=1) = \frac{1}{1 \cdot \sqrt{2\pi}} \approx 0.3989$$

When σ=2, the peak is:
$$f(0; \mu=0, \sigma=2) = \frac{1}{2 \cdot \sqrt{2\pi}} \approx 0.1995 = \frac{0.3989}{2}$$

This is **no coincidence** — the peak is proportional to $1/\sigma$. But when σ doubles, the curve also gets wider (its spread doubles). The two effects cancel exactly, so the total area always stays 1. Picture pouring a fixed amount of water: the wider you spread it, the shallower it gets, but the volume doesn't change.


### ⚠️ Core idea: probability density (PDF) vs probability (PMF)

Before going further with the normal distribution, let's clear up something that's easy to confuse: **why do continuous distributions use a "density" instead of giving probabilities directly?**

**Analogy: population and population density**

Picture how a country's population is spread out:
- The capital might have a population density of 500 people/km² (that's a "density" — one number)
- A small town might have 50 people/km²
- A desert might have 0.1 people/km²

If someone asks "what is the capital's population?", you can't answer "500" — a density is not a head count. You have to **multiply the density by the area** to get the actual number of people. For example, 1000 km² × 500 people/km² = 500,000 people.

**The probability density function works the same way:**

- **PDF f(x)** = "how concentrated the probability is near position x" (like population density)
- **f(0) ≈ 0.4** (the standard normal at its center) does not mean "P(X=0) = 0.4"
- It means "around x=0, the probability concentration is 0.4 per unit of length"

**How do you get an actual probability from a PDF?**

To find "P(a < X ≤ b)" (the probability that X lands in some interval), you need:

$$P(a < X \leq b) = \int_a^b f(x) \, dx$$

This "integrates" the density (or, put another way, takes a "weighted sum"). In the population example, it's "add up (density × width) slice by slice across [a,b]".

**Why can a PDF be > 1?**

A population density can be 1000 people/km² (far more than 1), and nobody finds that strange. Likewise, a probability density can be > 1. All that matters is that once you multiply it by a small enough width dx, f(x)·dx stays within [0,1].

**Key invariant: area = 1**

The PDF of the normal distribution integrates to 1 over the whole real line:

$$\int_{-\infty}^{\infty} f(x) \, dx = 1$$

This matches the plain fact that "the country's total population is fixed". However large σ is (however low the curve), as long as you multiply by the "magic constant" 1/(σ√2π) in front, the total area under the curve is always 1.


Before writing `gaussian_pdf`, pin down three things:
- Input: `x` (points to evaluate at, an array), `mu` (mean), `sigma` (standard deviation (SD), must be > 0)
- Key steps: compute the exponential term `exp(-(x-mu)**2 / (2*sigma**2))`, then divide by the normalizing constant `sigma * sqrt(2π)`
- Return: an array of PDF values the same shape as `x`, all non-negative

#### Maths formula vs code: why rewrite the exponent?

**The maths formula:**
$$\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

**The form in the code hint:**
```python
np.exp(-((x-mu)/sigma)**2 / 2)
```

**Are these really equal?** Let's check the algebra:

$$-\frac{(x-\mu)^2}{2\sigma^2} = -\frac{1}{2} \cdot \frac{(x-\mu)^2}{\sigma^2} = -\frac{1}{2} \left(\frac{x-\mu}{\sigma}\right)^2$$

So the two forms are exactly equivalent. Then why does the code use the second one?

**The reasons: numerical stability and simpler code**

1. **Avoid squaring big numbers**: when $|x-\mu|$ is large, $(x-\mu)^2$ becomes very large (though it only overflows at absurd magnitudes, which you'll rarely meet). $(x-\mu)/\sigma$, on the other hand, is usually a reasonable size; squaring it and dividing by 2 keeps the numbers better behaved and easier to read.

2. **Fewer moving parts**: `-((x-mu)/sigma)**2 / 2` is one tidy expression; `-(x-mu)**2 / (2*sigma**2)` has more nesting and more operations, which makes it easier to get wrong.

This is a common programming trick: **reorganize a maths expression so that the value doesn't change but the computation is more stable and easier to read**.


In [ ]:
def gaussian_pdf(x, mu=0.0, sigma=1.0):
    # ✏️ TODO: return the normal-distribution probability density at x
    raise NotImplementedError("Implement gaussian_pdf — see the line of reasoning above")


In [ ]:
import numpy as np

try:
    peak = gaussian_pdf(np.array([0.0]))
except (NotImplementedError, TypeError):
    print("⬜ Implement gaussian_pdf first, then run this cell")
else:
    peak_val = peak[0]
    assert abs(peak_val - 1/np.sqrt(2*np.pi)) < 1e-9, "the peak should be at μ"
    assert abs(gaussian_pdf(np.array([1.0]))[0] - gaussian_pdf(np.array([-1.0]))[0]) < 1e-12, "should be symmetric about μ"
    # check non-standard parameters mu=2, sigma=0.5
    y2 = gaussian_pdf(np.array([2.0]), mu=2.0, sigma=0.5)
    assert abs(y2[0] - 1/(0.5*np.sqrt(2*np.pi))) < 1e-9, "wrong peak for mu=2, sigma=0.5"
    # check normalization: with sigma=2 the integral should be ≈1
    xs_dense = np.linspace(-10, 10, 4000)
    integral = np.trapezoid(gaussian_pdf(xs_dense, 0.0, 2.0), xs_dense)
    assert abs(integral - 1.0) < 0.001, "with sigma=2 the integral should be ≈1"
    print("✅ Passed: Gaussian density peak =", round(peak_val, 4))


**🔗 Aurora connection**: why does the normal distribution matter so much?

`gaussian_pdf` is the most fundamental primitive in Aurora's generative pipeline. Here are three concrete uses:

1. **Weight initialization (He initialization)**
   - When a neural network starts training, its weights are W ~ N(0, √(2/n))
   - The parameter √(2/n) makes sure the signal neither explodes nor vanishes as it passes between layers (this is called "variance preservation")
   - `gaussian_pdf` gives the density of that distribution (in practice you usually only sample from it, but the theory needs the density)

2. **Latent-variable sampling in a VAE (variational autoencoder)**
   - The latent variable z ~ N(0, I) (standard normal)
   - The KL-divergence term of a VAE needs the density of N(0, I) to measure how far the posterior is from the prior
   - Here `gaussian_pdf` goes straight into the loss computation

3. **The noising process in diffusion models**
   - The noise added at each step is ε_t ~ N(0, β_t·I), where β_t is a time-dependent variance coefficient
   - `gaussian_pdf` is used to estimate the score function for the reverse (denoising) process

In all of these, the key steps involve:
- Given μ and σ, computing the probability density f(x) at a specific point x
- Sampling tensors from that distribution (with `np.random.normal` or PyTorch's `torch.randn`)
- Checking whether the sampled distribution satisfies orthogonality or other constraints

So **understanding the formula and parameters of `gaussian_pdf`** means understanding the basic maths of modern deep generative models.

In [ ]:
import numpy as np

# Fix mu=0, vary sigma, and watch how the peak and width change
xs = np.linspace(-4, 4, 100)
for sigma in [0.5, 1.0, 2.0]:
    try:
        vals = gaussian_pdf(xs, mu=0.0, sigma=sigma)
    except (NotImplementedError, TypeError):
        print("⬜ Implement gaussian_pdf first")
        break
    print(f'sigma={sigma:.1f}  peak={gaussian_pdf(np.array([0.0]), 0.0, sigma)[0]:.4f}')
    print(f'       area under the curve≈{np.trapezoid(vals, xs):.4f}  (should be ≈1.0)')


## Parameter experiment: turn one knob at a time

Fix `mu=0` and change `sigma` from 0.5 to 2: the peak drops from 0.8 to 0.2 (flatter), but the area under the curve is always 1. So sigma controls "how concentrated" the distribution is, without changing the total probability. Then change `mu` from 0 to 2: the whole curve shifts, but its shape (peak, width) doesn't change — mu only decides where the distribution is centered.

### From PDF to CDF: the cumulative view

If the PDF f(x) describes "how concentrated the probability is at a point", then the **CDF (cumulative distribution function)** is "the total probability, added up from the left, as far as the current position".

**Definition:**
$$F(x) = P(X \leq x) = \int_{-\infty}^{x} f(t) \, dt$$

**Analogy:** picture 24 hours of steady rain. f(t) is "the rainfall intensity in hour t" (mm/hour), and F(t) is "the total rainfall over the first t hours" (mm).

- F(6) - F(0) = "the rainfall in the first 6 hours"
- As t → 24, F(24) = "the total rainfall for the day" (the equivalent of an area of 1)

**Where does the 68-95-99.7 rule come from?**

We want to know "for the standard normal, how much probability lies within ±1σ", which means computing:

$$P(-1 < X < 1) = F(1) - F(-1) = \int_{-1}^{1} f(x) \, dx$$

But the normal PDF has no "closed-form" integral (F(x) can't be written with elementary functions), so we need a numerical method.

#### The trapezoidal rule: the basic method of numerical integration

**Core idea:** approximate the area under the curve with trapezoids.

Suppose we want $\int_a^b f(x) dx$:

1. Split the interval [a, b] into n equal pieces of width h = (b-a)/n
2. In each piece, instead of the exact curve, join the two endpoints with a "straight line" (making a trapezoid)
3. Trapezoid area = (top + bottom) × height / 2 = (f(x_i) + f(x_{i+1})) × h / 2
4. Add up all the trapezoid areas

**Simple example:** approximate the area of the normal distribution over [-2, 2] with n=4 trapezoids

| Interval | x_i | f(x_i) | Trapezoid area |
|---|---|---|---|
| [-2, -1] | -2, -1 | f(-2)=0.054, f(-1)=0.242 | (0.054+0.242)×1/2 ≈ 0.148 |
| [-1, 0] | -1, 0 | f(-1)=0.242, f(0)=0.399 | (0.242+0.399)×1/2 ≈ 0.320 |
| [0, 1] | 0, 1 | f(0)=0.399, f(1)=0.242 | (0.399+0.242)×1/2 ≈ 0.320 |
| [1, 2] | 1, 2 | f(1)=0.242, f(2)=0.054 | (0.242+0.054)×1/2 ≈ 0.148 |
| **Total** | | | ≈ 0.936 |

The theoretical value is 0.9545, so with n=4 the error is about 1.9%. Increasing n (more, thinner trapezoids) improves the accuracy.

#### How np.trapezoid works

```python
np.trapezoid(y, x)
```

where:
- `y` = the array of function values [f(x_0), f(x_1), ..., f(x_n)]
- `x` = the matching x coordinates [x_0, x_1, ..., x_n]

The function works out each trapezoid's width dx_i = x_{i+1} - x_i by itself, then sums:

$$\text{integral} ≈ \sum_{i=0}^{n-1} \frac{f(x_i) + f(x_{i+1})}{2} \cdot (x_{i+1} - x_i)$$

**Why this argument order?** Because the y values can come from any function (not necessarily a Gaussian), while x supplies the "ruler" (the positions). Given x, trapezoid knows the true width of every trapezoid.

#### Checking accuracy: how many trapezoids are enough?

A rule of thumb: to get the error below 0.1%, you usually need about n ≈ √(1 / target accuracy) points.

For the area of a normal distribution (target accuracy 0.001), n=2000 points is plenty. We can check:


In [ ]:
import numpy as np

# Check that the trapezoidal rule gets more accurate as the number of points grows
print("Checking the area within ±1σ for σ=1 (theory: 68.27%)\n")
print("points n |  integral  | error | verdict")
print("-" * 50)

theoretical_1sigma = 0.6827
for n in [10, 50, 200, 1000, 4000]:
    xs = np.linspace(-1, 1, n)
    try:
        pdf = gaussian_pdf(xs, mu=0.0, sigma=1.0)
    except (NotImplementedError, TypeError):
        print("⬜ Implement gaussian_pdf first")
        break

    area = np.trapezoid(pdf, xs)
    error = abs(area - theoretical_1sigma)
    status = "✓ accurate" if error < 0.001 else "△ rough"
    print(f"{n:5d}  |  {area:.6f}  | {error:.5f}  | {status}")

print("\n💡 Observation: the larger n, the more accurate the trapezoids; from n≥200 on it's accurate enough for engineering work")

### A closer look: why "the peak halves, but the area stays 1"

The experiment above showed something that looks like a contradiction:

- Change σ from 1 to 2: **the peak halves** (from 0.3989 → 0.1995)
- Yet **the area under the curve is still about 1**

How can that be? A "shorter" curve with the same area? The key is that **the curve also got wider at the same time**.

#### Checking the numbers: the growth in width cancels the drop in height

| σ | Peak | Width (±1σ range) | Approximate area | Note |
|---|---|---|---|---|
| 0.5 | 1/(0.5√2π) ≈ 0.798 | 1.0 ([-0.5, 0.5]) | ≈ 0.798 × 1.0 ≈ 0.80 | same product |
| 1.0 | 1/(1√2π) ≈ 0.3989 | 2.0 ([-1, 1]) | ≈ 0.3989 × 2.0 ≈ 0.80 | same product |
| 2.0 | 1/(2√2π) ≈ 0.1995 | 4.0 ([-2, 2]) | ≈ 0.1995 × 4.0 ≈ 0.80 | same product |

The table shows an intuitive relationship: **height × width (the area of a rough rectangle) stays constant** — about 0.80 here; the rectangle isn't the full area, but it doesn't change with σ.

Stated more precisely:

**Key insight: the peak is inversely proportional to σ**

$$\text{peak} = f(\mu) = \frac{1}{\sigma\sqrt{2\pi}} \propto \frac{1}{\sigma}$$

When σ doubles, the peak becomes $\frac{1}{2\sigma\sqrt{2\pi}}$, i.e. half of what it was.

**But the "width" of the curve doubles too.** What does "the width doubles" mean?

The precise statement is that "the amount of data within a given number of standard deviations" stays the same. For example, the range ±1σ always holds about 68.27% of the probability.

- When σ=1, the ±1σ range is [-1, 1] (width 2)
- When σ=2, the ±1σ range is [-2, 2] (width 4)

In numerical integration, when the interval width doubles and the height halves at the same time, the area from the trapezoidal rule (a rectangle approximation) stays the same. This is no coincidence — it's exactly what the $\frac{1}{\sigma}$ factor in the normal formula is designed to do.

#### Checking in code: measure the area directly

Run the code below and see how different σ values affect the integral:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Check that the area is conserved
xs_wide = np.linspace(-10, 10, 2000)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for sigma in [0.5, 1.0, 2.0]:
    try:
        pdf = gaussian_pdf(xs_wide, mu=0.0, sigma=sigma)
    except (NotImplementedError, TypeError):
        print("⬜ Implement gaussian_pdf first")
        break

    area = np.trapezoid(pdf, xs_wide)
    peak = gaussian_pdf(np.array([0.0]), 0.0, sigma)[0]

    # left: look at the shape of the curve
    ax1.plot(xs_wide, pdf, label=f'σ={sigma:.1f} (peak={peak:.4f}, area={area:.4f})', linewidth=2)

    # right: approximate area (rectangle height × width)
    approx_rect_area = peak * (2 * sigma)  # simplification: peak × width
    ax2.bar(sigma, area, width=0.15, label=f'integrated area', alpha=0.7)
    ax2.bar(sigma + 0.08, approx_rect_area, width=0.15, label=f'rough rectangle', alpha=0.7)

ax1.set_xlabel('x'); ax1.set_ylabel('PDF f(x)'); ax1.set_title('Normal curves for different σ')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_ylabel('area'); ax2.set_xlabel('σ'); ax2.set_title('Area conservation: height × width = constant')
ax2.set_ylim([0, 1.5]); ax2.axhline(1.0, color='red', linestyle='--', label='expected area = 1')
ax2.legend(); plt.tight_layout(); plt.show()

print("✅ Observation: whatever σ is, the area is always about 1 (height and width move in opposite directions)")

In [ ]:
import numpy as np

# CDF of the normal distribution: approximated by numerical integration
xs = np.linspace(-4, 4, 1000)
for sigma in [0.5, 1.0, 2.0]:
    try:
        pdf = gaussian_pdf(xs, 0.0, sigma)
    except (NotImplementedError, TypeError):
        print("⬜ Implement gaussian_pdf first")
        break
    cdf = np.cumsum(pdf) * (xs[1] - xs[0])
    # rule of thumb: μ±1σ covers ≈68% of the area
    mask = np.abs(xs) <= sigma
    area_1sigma = np.trapezoid(pdf[mask], xs[mask])
    print(f'sigma={sigma:.1f}  area within μ±1σ={area_1sigma:.3f}  (expected ≈0.683)')


## 3. The Bernoulli distribution

A single yes/no trial: success (X=1) with probability `p`, failure (X=0) with probability `1-p`.

**PMF (probability mass function)**:
```
P(X=k) = p^k * (1-p)^(1-k),  k ∈ {0,1}
```

**Expectation** = `p`, **variance** = `p(1-p)`.

**Aurora connection**: each element of a dropout mask follows Bernoulli(p=1-dropout_rate), deciding independently whether to keep that neuron. SpecAugment, used in audio data augmentation, also uses Bernoulli masks to blank out random frequency bands.

#### Where the Bernoulli PMF formula comes from, and why it's elegant

The most naive way to write it is case by case:
$$P(X=k) = \begin{cases} 1-p & \text{if } k=0 \\ p & \text{if } k=1 \end{cases}$$

But if you want one unified formula (rather than branches), you can cleverly use **the exponent as a "switch"**:

$$P(X=k) = p^k (1-p)^{1-k}$$

**Why does this formula work?**

Plug in and check:
- k=0: $p^0 (1-p)^{1-0} = 1 \cdot (1-p) = 1-p$ ✓
- k=1: $p^1 (1-p)^{1-1} = p \cdot 1 = p$ ✓

**How would anyone think of this "magic"?**

Think of k as an "exponent selector":
- When k=0, the first factor $p^0$ becomes 1 ("off") and the second factor $(1-p)^1$ becomes 1-p ("on")
- When k=1, the first factor $p^1$ becomes p ("on") and the second factor $(1-p)^0$ becomes 1 ("off")

The advantages of writing it this way:
1. **Compact**: one formula instead of an if-else
2. **Easy to generalize**: for a multinomial distribution (several options) it extends naturally to $\prod_i p_i^{n_i}$
3. **Easy to compute with**: in log space, $\log P = k \log p + (1-k) \log(1-p)$, and addition is more stable than multiplication

#### Computing the Bernoulli expectation and variance

Straight from the PMF:
- **Expectation (expected value)** = $E[X] = 0 \cdot (1-p) + 1 \cdot p = p$
- **Variance (a measure of spread)** = $\text{Var}(X) = E[X^2] - (E[X])^2 = p - p^2 = p(1-p)$

Interestingly, when p=0.5 (the most "uncertain" case), the variance is at its maximum = 0.25. When p is close to 0 or 1 (the outcome is nearly certain), the variance is close to 0.


### PDF vs PMF: two kinds of distribution function

So far, the **normal distribution we've studied is continuous** — X can take any real value (3.14159... is fine). But in the real world there are also **discrete distributions** — X can only take particular values (say 0 or 1).

| Property | PDF (continuous) | PMF (discrete) |
|---|---|---|
| **Domain** | real numbers (infinitely many possible values) | finitely or countably many values |
| **Function values** | can be > 1 (a density) | always ≤ 1 (a probability) |
| **Summing up** | by integration: ∫ | by summation: Σ |
| **Normalization** | ∫ f(x) dx = 1 | Σ P(X=k) = 1 |
| **Examples** | normal, uniform | Bernoulli, Poisson |

**Intuition:**
- "What's the probability of rolling a 3?" Use the PMF: the answer is 1/6 (a definite number)
- "What's the probability that a person's height is between 170 and 180 cm?" Use the PDF: the answer needs an "integral" (because height varies continuously)

**Why doesn't the Bernoulli distribution use a PDF?**

A Bernoulli random variable X ∈ {0, 1} has only two possible values. You can't describe it with a "density" (a density can't be defined at 0 and 1); you can only use "probability mass".

An analogy:
- PDF: imagine spreading 1 unit of "sand" continuously along the real line; the PDF describes "how thick the sand is at each spot"
- PMF: put the 1 unit of "sand" on two points (X=0 and X=1); the PMF describes "how much sand sits on each point"


In [ ]:
import numpy as np

# Bernoulli PMF
def bernoulli_pmf(k, p):
    k = np.asarray(k)
    return (p ** k) * ((1 - p) ** (1 - k))

# Check the PMF: the probabilities for k=0 and k=1 sum to 1
for p in [0.3, 0.5, 0.8]:
    pmf0 = bernoulli_pmf(0, p)
    pmf1 = bernoulli_pmf(1, p)
    print(f'p={p}: P(X=0)={pmf0:.3f}, P(X=1)={pmf1:.3f}, sum={pmf0+pmf1:.3f}')
    assert abs(pmf0 + pmf1 - 1.0) < 1e-9

# Check by sampling: the mean of a large sample approaches p
rng = np.random.default_rng(0)
for p in [0.3, 0.5, 0.8]:
    samples = rng.binomial(1, p, size=100_000)
    print(f'p={p}: sample mean={samples.mean():.4f}  variance={samples.var():.4f}  theoretical variance={p*(1-p):.4f}')

print('✅ Bernoulli PMF and sample-mean checks passed')


## 4. The cumulative distribution function (CDF)

CDF definition: `F(x) = P(X ≤ x)` — the probability that the random variable takes a value no greater than x.

| Distribution | CDF |
|---|---|
| Uniform U[a,b] | F(x) = (x-a)/(b-a) for x∈[a,b] |
| Normal N(μ,σ²) | by numerical integration: ∫ pdf(t)dt |
| Bernoulli B(p) | F(0)=1-p, F(1)=1 |

**Properties**: a CDF never decreases, F(-∞)=0, F(+∞)=1; the CDF of a continuous distribution is continuous everywhere.

**Use**: given a CDF, the probability of any interval [a,b] is just a subtraction: `P(a < X ≤ b) = F(b) - F(a)`.

### Why can't the CDF be computed with a closed-form formula?

The normal PDF has a neat formula, but **the CDF doesn't**. What does that mean?

$$F(x) = \int_{-\infty}^{x} \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(t-\mu)^2}{2\sigma^2}\right) dt$$

The integral looks simple, but **it has no answer expressible in elementary functions (polynomials, trig functions, logarithms and so on)**. This was proved in the 19th century — a mathematical "dead end".

By contrast:
- Uniform CDF: $F(x) = \frac{x-a}{b-a}$ (a straight line, one formula)
- Bernoulli CDF: piecewise constant (0, then 1-p, then 1)
- Normal CDF: **has to be approximated numerically**

#### Implementing the normal CDF numerically: where do we start integrating?

The definition is $F(x_{\text{eval}}) = \int_{-\infty}^{x_{\text{eval}}} f(t) dt$

But a computer can't handle "infinity", so we replace $-\infty$ with a far-left endpoint $x_{\text{left}} = \mu - 6\sigma$.

**Why 6σ?**

From the 68-95-99.7 rule:
- the range ±3σ holds 99.73% of the probability
- so the probability beyond μ-6σ is < (1-0.9973)/2 = 0.00135 ≈ 0.1%

When we start integrating at μ-6σ, the part we ignore is at most 0.1%, which is accurate enough for most purposes.

What if we used μ-3σ or μ-10σ instead?
- μ-3σ: ignores 0.135%, which may not be quite accurate enough
- μ-10σ: ignores almost 0%, but the range is so wide that it wastes computation

**Closed form vs numerical method — the trade-off:**

| Property | Closed-form formula | Numerical integration |
|---|---|---|
| Always expressible as a formula? | no (the normal CDF has none) | always possible |
| Speed | extremely fast (direct evaluation) | slower (needs a loop) |
| Accuracy control | floating-point precision (≈15 digits) | freely adjustable (more trapezoids → more accurate) |
| Scope | only a few special distributions | can integrate any function |

So the normal CDF is always computed numerically, which is also why `scipy.stats.norm.cdf()` uses an approximation algorithm under the hood (a Taylor series for the error function, or table lookup).


In [ ]:
import numpy as np

# Normal CDF (numerical integration: trapezoidal rule)
def normal_cdf(x_eval, mu=0.0, sigma=1.0, n=2000):
    xs = np.linspace(mu - 6*sigma, x_eval, n)
    pdf = np.exp(-0.5*((xs - mu)/sigma)**2) / (sigma * np.sqrt(2*np.pi))
    return np.trapezoid(pdf, xs)

# Check the rule of thumb: N(0,1) lands in [-1,1] with probability ≈ 68.27%
p_within_1sd = normal_cdf(1.0) - normal_cdf(-1.0)
p_within_2sd = normal_cdf(2.0) - normal_cdf(-2.0)
p_within_3sd = normal_cdf(3.0) - normal_cdf(-3.0)

print(f'P(-1 < X < 1) = {p_within_1sd:.4f}  (theory 68.27%)')
print(f'P(-2 < X < 2) = {p_within_2sd:.4f}  (theory 95.45%)')
print(f'P(-3 < X < 3) = {p_within_3sd:.4f}  (theory 99.73%)')

assert abs(p_within_1sd - 0.6827) < 0.002
assert abs(p_within_2sd - 0.9545) < 0.002
assert abs(p_within_3sd - 0.9973) < 0.002
print("✅ Normal CDF rule-of-thumb check passed")


## Wrap-up

You can now use `gaussian_pdf(x, mu, sigma)` to get the density of a normal distribution at any point, and use `np.random.uniform` / `np.random.normal` to draw samples from the two distributions. `gaussian_pdf` corresponds to the density evaluation in Aurora's noising step and to the KL-divergence computation in a VAE. Next lesson uses softmax to turn unnormalized scores into a probability distribution, and the idea of a normalizing constant you wrote by hand today will show up there again.

---
⬇️ **Pass check**: you've read the wrap-up; finish the whiteboard challenge below before ticking off the self-review.


### An important property: the normal distribution is symmetric about μ

One line in the code checks "symmetric about μ":
```python
assert abs(gaussian_pdf(np.array([1.0]))[0] - gaussian_pdf(np.array([-1.0]))[0]) < 1e-12
```

This tests f(1) ≈ f(-1) (when μ=0). More generally, the normal distribution satisfies:

$$f(\mu + a) = f(\mu - a)$$

for every a > 0. This is called being "symmetric about μ".

**Why does it hold?** Look at $(x-\mu)^2$ in the formula:
- When x = μ+1: $(μ+1-μ)^2 = 1^2 = 1$
- When x = μ-1: $(μ-1-μ)^2 = (-1)^2 = 1$

The property of squares $(a)^2 = (-a)^2$ guarantees the symmetry.

**What physical intuition does this match?** The "shape" of the normal distribution is a bell: high in the middle, low on both sides. Whether you look left or right from the center, the pattern is the same. This matches the symmetry of "measurement error": overestimating and underestimating are equally likely.

**Are there asymmetric distributions?** Yes. For example, "students' reading speed" may not be symmetric (fast at first, slower later), and then you need a "skewed distribution" (such as the log-normal) to model it. The normal distribution's symmetry is one of its special advantages.


## ✏️ Whiteboard challenge: probability distributions by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: for the standard normal distribution (μ=0, σ=1), what is the peak f(0)?
(Formula: 1/(σ√2π))

**Question 2**: when σ increases from 1 to 2, what does the peak become? (What is its ratio to the answer to Question 1?)

> **Digging deeper (optional, not needed to pass)**: what if you want to compute by hand not the peak f(0) but any other point on the curve, say f(0.5)?
>
> Don't panic — the steps are the same as for the peak, with one extra "standardize" step:
> 1. First compute the "standardized distance" $z = \dfrac{x-\mu}{\sigma}$. Here $x=0.5,\ \mu=0,\ \sigma=1$, so $z=0.5$.
> 2. Compute the exponential part $e^{-z^2/2} = e^{-0.125}$. To work out $e^{-0.125}$ by hand you can use the approximation $e^{-t}\approx 1-t+\frac{t^2}{2}$ (good enough when t is small): $1-0.125+0.0078\approx 0.883$.
> 3. Multiply by the normalizing constant $\frac{1}{\sqrt{2\pi}}\approx 0.399$: $0.399\times0.883\approx0.352$.
>
> So $f(0.5;0,1)\approx 0.352$ — a little lower than the peak of 0.399, which fits the intuition of a bell curve falling away on both sides of the peak. The answer-check cell below verifies this hand calculation in code.

**Question 3**: Bernoulli distribution (p=0.3):
- P(X=1) = ?
- E[X] = ?
- Var[X] = p(1-p) = ?

**Question 4**: by the 68-95-99.7 rule for the normal distribution, roughly what % of the area does μ±2σ cover?

**Question 5 (application)**: MFCC features and the normal distribution

> MFCCs (mel-frequency cepstral coefficients) are an important feature in audio signal processing; they mimic how the human auditory system perceives sound. (L01 introduced them and L50 builds the extraction step by step; here we only care about their statistics.)
>
> In practice, MFCC features are often standardized: if the MFCC features are distributed with mean = 0 and standard deviation = 1, then by the 68-95-99.7 rule, in what range will the vast majority (>99%) of the feature values fall?

When you've worked them out, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: f(0; μ=0, σ=1) = 1/√(2π) ≈ 0.3989
peak_expected = 1 / np.sqrt(2 * np.pi)
assert np.isclose(peak_expected, 0.3989422804014327, atol=1e-10)
try:
    peak_computed = gaussian_pdf(np.array([0.0]), mu=0.0, sigma=1.0)[0]
    assert np.isclose(peak_computed, peak_expected, atol=1e-10)
    print(f"Q1 ✅  f(0;0,1)=1/√2π={peak_expected:.6f}, your function gives {peak_computed:.6f}")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement gaussian_pdf() first, then run the answer-check cell")

# Q2: with σ=2 the peak halves
peak_sigma2 = 1 / (2 * np.sqrt(2 * np.pi))
ratio = peak_sigma2 / peak_expected
assert np.isclose(ratio, 0.5, atol=1e-12)
print(f"Q2 ✅  peak at σ=2 is {peak_sigma2:.6f}, {(1/ratio):.0f}× smaller than at σ=1 (the secret of the halving: height ∝ 1/σ, width ∝ σ, height × width = constant)")

# Q2, digging deeper: a non-peak point f(0.5) by hand
try:
    f_half_computed = gaussian_pdf(np.array([0.5]), mu=0.0, sigma=1.0)[0]
    f_half_handcalc = (1 - 0.125 + 0.125**2 / 2) / np.sqrt(2 * np.pi)  # hand approximation of e^-0.125
    f_half_exact = np.exp(-0.5 * 0.5**2) / np.sqrt(2 * np.pi)
    print(f"Q2 deeper ✅  f(0.5) by hand ≈ {f_half_handcalc:.4f}, exact = {f_half_exact:.4f}, your function = {f_half_computed:.4f}")
except (NotImplementedError, TypeError):
    print("⬜ Q2 deeper: implement gaussian_pdf() first, then run the answer-check cell")

# Q3: Bernoulli(p=0.3)
p = 0.3
px1 = p                    # P(X=1) = p^1 * (1-p)^0 = p
e_x = p                    # E[X] = p
var_x = p * (1-p)          # Var[X] = p(1-p)
assert np.isclose(px1, 0.3) and np.isclose(e_x, 0.3) and np.isclose(var_x, 0.21)
print(f"Q3 ✅  Bernoulli(0.3): P(X=1)={px1}, E[X]={e_x}, Var[X]={var_x:.2f}")

# Q4: the 68-95-99.7 rule
rng = np.random.default_rng(0)
samples = rng.normal(0, 1, 1_000_000)
within_2sigma = np.mean(np.abs(samples) <= 2)
assert abs(within_2sigma - 0.9545) < 0.005, f"μ±2σ should cover ≈95.45%, got {within_2sigma:.4f}"
within_3sigma = np.mean(np.abs(samples) <= 3)
assert within_3sigma > 0.99
print(f"Q4 ✅  μ±2σ covers {within_2sigma:.4f}≈95.45%, μ±3σ covers {within_3sigma:.4f}>99%")

# Q5: the range of MFCC features (μ=0, σ=1)
print(f"\nQ5 ✅  MFCC features (μ=0, σ=1): 99%+ of the values fall in [-3, 3]")
print(f"     (more precisely, 99.73% lie within ±3σ, so roughly 27 in every 10000 samples fall outside)")
print(f"\n     💡 This tells us: if a new MFCC value is > 3 or < -3,")
print(f"       it's a 'rare case' — possibly noise or an unusual audio event.")
print(f"\n     ⚠️ But note: if you actually run the experiment 'draw 1000 MFCC values at random',")
print(f"       the number outside won't be exactly 2.7 — it wobbles at random between about 0 and 5;")
print(f"       that's 'sampling variation', not a calculation error (see the extended discussion and code in the law-of-large-numbers section).")

print("\n🎉 Probability-distribution whiteboard challenge passed! Normal, uniform and Bernoulli are now second nature.")

In [ ]:
# ✏️ Self-review for this lesson
l29_review = {
    "gaussian_pdf_formula":       None,  # remember the normal PDF formula? True/False
    "gaussian_pdf_implemented":   None,  # gaussian_pdf implemented and assertions pass? True/False
    "sigma_peak_relationship":    None,  # understand σ↑ → peak↓ but area = 1? True/False
    "rule_68_95_997":             None,  # remember the 68-95-99.7 rule? True/False
    "whiteboard_passed":          None,  # whiteboard challenge done on paper? True/False
}

unfilled = [k for k, v in l29_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l29_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L29 fully cleared! On to L30: Softmax and cross-entropy')

In [ ]:
# Quick check: μ±1σ coverage from large samples (direct sampling, no gaussian_pdf needed)
# a normal distribution's μ±1σ covers about 68.27% of the area
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    samples = rng.normal(0, 1, size=n)
    within_1std = np.sum(np.abs(samples) <= 1.0) / n
    print(f'n={n:4d} -> share of samples within μ±1σ={within_1std:.3f}  (theory ≈0.683)')


---

→ **Next lesson** · [L30 · Softmax and cross-entropy](L30_softmax_crossentropy.ipynb)

> Next lesson covers **softmax and cross-entropy**: the output layer and loss of a classifier, with the gradient derived by hand.